In [150]:
import csv
import numpy as np

In [151]:
def load_data():
    with open("data.csv") as f:
        reader=csv.reader(f)
        data=np.array(list(reader),dtype=float)
        return data

In [152]:
print(load_data())

[[  6.    148.     72.    ...   0.627  50.      1.   ]
 [  1.     85.     66.    ...   0.351  31.      0.   ]
 [  8.    183.     64.    ...   0.672  32.      1.   ]
 ...
 [  5.    121.     72.    ...   0.245  30.      0.   ]
 [  1.    126.     60.    ...   0.349  47.      1.   ]
 [  1.     93.     70.    ...   0.315  23.      0.   ]]


In [153]:
def split_data(data,x):
    len_data=len(data) ## 768
    split_idx=int(len_data*x) ## suppose x=0.7, extract 70% data to train model and 30% to test trained model 
    training_data=data[:split_idx]
    testing_data=data[split_idx:]
    return training_data,testing_data

In [154]:
def split_values_results(data):
    values=data[:,:-1] ##all rows's vlaues except last column (which is either 0 or 1)
    results=data[:,-1] ## all result values (each is 0->no diabetes or 1->yes diabetes)
    # print(values)
    # print(results)
    return values,results

In [163]:
import math
def gausian_probability(x,mean,std_deviation):
    dr=math.sqrt(2*(math.pi)*(std_deviation**2))
    nr=math.exp(
            -((x-mean)**2)/
              (2*(std_deviation**2))
             )
    return nr/dr

In [164]:
def TRAIN_MODEL(values,results):
    possible_results=np.unique(results)
    print("possible results are : ",possible_results) ## here it is [0,1] )-> no diabetes, 1->yes diabetes
    no_diabetes_data=[]
    yes_diabetes_data=[]
    for i in range(len(values)):
        row=values[i]
        its_res=results[i]
        if its_res==0:
            no_diabetes_data.append(row)
        else:
            yes_diabetes_data.append(row)

    ## no compute mean and std deviation for yes and no diabetees data, axis=0, means calc mean col wise, if not mentionsed it will flatten 2d matrix into 1d and calc one single mean value
    no_diabetes_mean_of_each_col,no_diabetes_std_dev_of_each_col = np.mean(no_diabetes_data, axis=0),np.std(no_diabetes_data,axis=0)
    yes_diabetes_mean_of_each_col,yes_diabetes_std_dev_of_each_col = np.mean(yes_diabetes_data, axis=0),np.std(yes_diabetes_data,axis=0)

    return {
        0:{'mean':no_diabetes_mean_of_each_col,'std_dev':no_diabetes_std_dev_of_each_col},
        1:{'mean':yes_diabetes_mean_of_each_col,'std_dev':yes_diabetes_std_dev_of_each_col}
    }

In [169]:
def TEST(values, results, no_dia_mean, no_dia_std_dev, yes_dia_mean, yes_dia_std_dev):
    correct = 0
    total = len(values)

    for i in range(total):
        row = values[i]

        prob_0 = 1
        prob_1 = 1

        # calculate probability for each feature
        for j in range(len(row)):
            prob_0 *= gausian_probability(row[j], no_dia_mean[j], no_dia_std_dev[j])
            prob_1 *= gausian_probability(row[j], yes_dia_mean[j], yes_dia_std_dev[j])

        # predict class
        predicted = 0 if prob_0 > prob_1 else 1

        # compare with actual
        if predicted == results[i]:
            correct += 1

    accuracy = (correct / total) * 100
    return accuracy

In [172]:
data=load_data()
training_data,test_data=split_data(data,0.7)

## Phase 1: traiing model: first segrate data based on 0 and 1 result, for each then calc mean and std dev of each col 
training_x_values,training_y_values=split_values_results(training_data)
model=TRAIN_MODEL(training_x_values,training_y_values)

print("mean and std deviation of O(no diabetes) and 1(yes diabetes) of each col is: ")
print("no diabetes's mean and std_dev: \nMean=",model[0]['mean'],"\nStd_Dev=",model[0]['std_dev'])
print("\n yes diabetes's mean and std_dev: \nMean=",model[1]['mean'],"\nStd_Dev=",model[1]['std_dev'])


## Phase 2: testing our trained model
testing_x_values,testing_y_values=split_values_results(test_data)
accuracy=TEST(testing_x_values,testing_y_values,model[0]['mean'],model[0]['std_dev'],model[1]['mean'],model[1]['std_dev'])
print("accuracy",accuracy)

possible results are :  [0. 1.]
mean and std deviation of O(no diabetes) and 1(yes diabetes) of each col is: 
no diabetes's mean and std_dev: 
Mean= [  3.27873563 109.87931034  67.89655172  19.53735632  67.47701149
  29.94511494   0.43953736  31.30747126] 
Std_Dev= [  2.97304105  27.23466198  17.90959704  14.79209504 102.16114181
   7.99027097   0.30280953  11.82415468]

 yes diabetes's mean and std_dev: 
Mean= [  4.83068783 139.6984127   69.56613757  21.7037037  100.53968254
  35.25396825   0.56379894  36.35978836] 
Std_Dev= [  3.76321798  32.55892579  22.4226387   17.10957642 138.44260271
   7.37726004   0.38844351  10.65364642]
accuracy 77.48917748917748
